# 01 基準計測（Phase 2）
現行 lv5 の対戦・ひとり用の基準値と、学習1世代あたりの所要時間を測る。
設計: `source_assets/memory/cpu/puyo/lv5-training-design.md`

In [ ]:
import sys, subprocess, time, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "py"))
import numpy as np
import puyosim as ps
ps.build(force=True)  # 本番 native ソースから共有ライブラリを作り直す

## JS エンジンとの一致テスト（Phase 1 の受け入れ条件）

In [ ]:
print(subprocess.run([sys.executable, str(ROOT / "tests/test_parity.py")], capture_output=True, text=True).stdout)
print(subprocess.run(["bash", str(ROOT / "tests/run_cpp_tests.sh")], capture_output=True, text=True).stdout)

## 現行 lv5 同士（先後入れ替え）：1局の長さ・スループット

In [ ]:
base = ps.Policy.baseline().packed()
seeds = np.arange(1, 101, dtype=np.uint32) * 4999 % 499999 + 1
t0 = time.time()
r = ps.play_both_sides(base, base, seeds)
sec = time.time() - t0
games = len(seeds) * 2
raw = r["raw"]
print(f"{games} 局 / {sec:.1f}s = {games / sec:.2f} 局/秒")
print(f"ゲーム内時間 平均 {raw['durationMs'].mean() / 1000:.1f}s, 手数 平均 {raw['turnsA'].mean():.1f}")
print(f"最大連鎖 平均 {raw['maxChainA'].mean():.2f}, 引き分け率 {np.mean(raw['winner'] == -1):.3f}")
print(f"単局の先手勝率 {np.mean(raw['winner'] == 0):.3f}（同時刻イベントの先手優遇。学習では先後2局で打ち消す）")

In [ ]:
# 学習1世代の所要時間見積もり（候補数 × シード数 × 2局）
pop, nseed = 16, 100
print(f"1世代 ≈ {pop * nseed * 2 / (games / sec) / 60:.1f} 分, 100世代 ≈ {pop * nseed * 2 / (games / sec) * 100 / 3600:.1f} 時間")

## ひとり用の構築指標（60手・おじゃま無し）

In [ ]:
solo = ps.run_solo(base, np.arange(1, 301, dtype=np.uint32), max_turns=60, target_chain=10)
reached = solo["firstBigTurn"] >= 0
print(f"最大連鎖 平均 {solo['maxChain'].mean():.2f} / 10連鎖到達率 {reached.mean():.2f} "
      f"/ 到達手数 平均 {solo['firstBigTurn'][reached].mean():.1f} / 窒息率 {solo['died'].mean():.2f}")